# Building a Gaussian Process Emulator for the Halo Mass Function
## Summer Student Program — Solutions Notebook

### Goal
Cosmological simulations are expensive. An **emulator** is a fast surrogate model that learns the output of an expensive calculation (here, the halo mass function, HMF) as a function of input parameters (here, cosmological parameters), so that we can evaluate it in milliseconds instead of hours.

In this notebook you will build a complete emulator pipeline from scratch:

1. Use **Dark Emulator** as our "expensive simulation" (it is actually fast, which makes it ideal for experimenting).
2. Design a training set with **Latin Hypercube Sampling** (LHS).
3. Compress the HMF curves with **Principal Component Analysis** (PCA).
4. Fit a **Gaussian Process** (GP) to each PCA coefficient as a function of cosmology.
5. **Validate** the emulator against the truth and **improve** it.

### How to use this notebook
- Cells marked `# TODO` have blanks for you to fill in (`...`).
- Questions marked **Q** ask you to think and write a short answer in the markdown cell below.
- Try to run cells in order. Later cells depend on earlier variables.

### Requirements
```
pip install dark_emulator scikit-learn scipy matplotlib numpy
```


---
## Part 0 — Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dark_emulator import darkemu

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['font.size'] = 12

---
## Part 1 — The "truth": the halo mass function from Dark Emulator

Dark Emulator predicts the number of dark-matter halos in a mass bin $[M_1, M_2]$ for a given volume and redshift, as a function of six cosmological parameters:

$$\mathbf{p} = (\omega_b,\ \omega_c,\ \Omega_\Lambda,\ \ln(10^{10} A_s),\ n_s,\ w)$$

Throughout this notebook we will **only vary two parameters**, $\Omega_m$ and $\ln A_s$, and fix the rest to the values below. Note that Dark Emulator takes $\Omega_\Lambda = 1 - \Omega_m$ (flat universe).

Let's first compute and plot one HMF at the fiducial cosmology.


In [ ]:
emu = darkemu.base_class()

z = 0.0            # redshift
vol = 1e9          # volume in (Mpc/h)^3
log10M_min, log10M_max, n_bins = 12.0, 15.0, 50

log10M = np.linspace(log10M_min, log10M_max, n_bins)     # bin edges
log10M_mid = 0.5 * (log10M[:-1] + log10M[1:])            # bin centres
nbin = len(log10M) - 1

# fixed parameters
omega_b, omega_c, n_s, w = 0.02225, 0.1198, 0.9645, -1.0

def get_hmf_emu(Om, lnAs):
    """Return the number of halos in each mass bin for a given (Omega_m, lnAs)."""
    cparam = np.array([omega_b, omega_c, 1.0 - Om, lnAs, n_s, w])
    emu.set_cosmology(cparam)
    return np.array([emu.get_nhalo(10**log10M[i], 10**log10M[i+1], vol, z) for i in range(nbin)])

hmf_fid = get_hmf_emu(Om=0.3, lnAs=3.1)

plt.semilogy(log10M_mid, hmf_fid, 'o-')
plt.xlabel(r'$\log_{10} (M\,[h^{-1}M_\odot])$')
plt.ylabel(r'$N_{\rm halo}$ per bin')
plt.title('HMF at fiducial cosmology')
plt.grid(True, which='both', ls=':')

### Exercise 1.1
Plot the HMF for three values of $\Omega_m$ (e.g. 0.2, 0.3, 0.4) at fixed $\ln A_s = 3.1$, and separately for three values of $\ln A_s$ (e.g. 2.6, 3.1, 3.6) at fixed $\Omega_m = 0.3$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for Om in [0.2, 0.3, 0.4]:
    axes[0].plot(log10M_mid, get_hmf_emu(Om, 3.1), label=rf'$\Omega_m={Om}$')
axes[0].set_title(r'varying $\Omega_m$ at $\ln A_s = 3.1$')

for lnAs in [2.6, 3.1, 3.6]:
    axes[1].plot(log10M_mid, get_hmf_emu(0.3, lnAs), label=rf'$\ln A_s={lnAs}$')
axes[1].set_title(r'varying $\ln A_s$ at $\Omega_m = 0.3$')

for ax in axes:
    ax.set_yscale('log'); ax.set_xlabel(r'$\log_{10} M$'); ax.legend(); ax.grid(True, which='both', ls=':')
axes[0].set_ylabel(r'$N_{\rm halo}$')

**Q1.1** Which parameter changes the *shape* of the HMF more, and which changes the *amplitude*? Why does the high-mass end respond more strongly than the low-mass end?

*Your answer:*


---
## Part 2 — Designing the training set: Latin Hypercube Sampling

To train an emulator we need to evaluate the "simulation" at a set of input parameters. Random sampling leaves gaps and clusters; a regular grid needs $N^d$ points in $d$ dimensions. **Latin Hypercube Sampling** spreads points so that every 1D projection is uniformly covered.

We will sample in the box
$$\Omega_m \in [0.2, 0.4], \qquad \ln A_s \in [2.5, 3.7].$$


In [ ]:
from scipy.stats import qmc

n_params = 2
bounds = np.array([[0.2, 0.4],     # Omega_m
                   [2.5, 3.7]])    # lnAs
n_samples = 100

sampler = qmc.LatinHypercube(d=n_params, seed=42)
unit_samples = sampler.random(n=n_samples)
train_params = qmc.scale(unit_samples, bounds[:, 0], bounds[:, 1])

print(train_params.shape)

### Exercise 2.1
Compare the LHS design against plain uniform random sampling with the same number of points. Plot both designs side by side, and add histograms of the $\Omega_m$ values (1D projection) for each.


In [ ]:
rng = np.random.default_rng(0)
random_params = rng.uniform(bounds[:, 0], bounds[:, 1], size=(n_samples, 2))

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, (name, pts) in enumerate([('Latin Hypercube', train_params), ('Uniform random', random_params)]):
    axes[0, col].plot(pts[:, 0], pts[:, 1], 'o', ms=4)
    axes[0, col].set_xlabel(r'$\Omega_m$'); axes[0, col].set_ylabel(r'$\ln A_s$'); axes[0, col].set_title(name)
    axes[1, col].hist(pts[:, 0], bins=20, range=bounds[0], edgecolor='k')
    axes[1, col].set_xlabel(r'$\Omega_m$'); axes[1, col].set_ylabel('count')
plt.tight_layout()

**Q2.1** Why is a uniform 1D projection desirable for an emulator? What would happen with a regular grid if we had 6 parameters and wanted 10 points per axis?

*Your answer:*


---
## Part 3 — Building the training set

Now evaluate the "truth" at each design point. Store the results in a matrix `hmfs` of shape `(n_samples, nbin)`.

> **Design choice:** the HMF spans many orders of magnitude. We will emulate $\log_{10} N_{\rm halo}$ rather than $N_{\rm halo}$ itself. You will explore why in Exercise 5.2.


In [ ]:
hmfs = np.zeros((n_samples, nbin))
for i in range(n_samples):
    hmfs[i] = get_hmf_emu(train_params[i, 0], train_params[i, 1])

# emulate in log space
Y_train = np.log10(hmfs)

plt.semilogy(log10M_mid, hmfs.T, color='C0', alpha=0.2)
plt.xlabel(r'$\log_{10} M$'); plt.ylabel(r'$N_{\rm halo}$'); plt.title('All training HMFs')

---
## Part 4 — Compressing the output with PCA

Each HMF is a vector of 49 numbers, but the curves are highly correlated: they are all controlled by just two parameters. PCA finds a small set of basis curves ("modes") such that

$$ \log_{10} N(\mathbf{p}) \approx \bar{y} + \sum_{k=1}^{n_{\rm pc}} a_k(\mathbf{p})\, \phi_k ,$$

where $\phi_k$ are the principal components and $a_k$ are the coefficients. Instead of emulating 49 numbers we then only need to emulate $n_{\rm pc}$ coefficients.

We first standardise each mass bin (zero mean, unit variance) so that bins with large dynamic range do not dominate.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler().fit(Y_train)
Y_scaled = scaler.transform(Y_train)

pca = PCA(n_components=0.9999, svd_solver='full')
pc_scores = pca.fit_transform(Y_scaled)
n_pc = pc_scores.shape[1]

print("number of PCs kept:", n_pc)
print("explained variance ratio:", pca.explained_variance_ratio_)
print("cumulative:", np.cumsum(pca.explained_variance_ratio_))

### Exercise 4.1
Make (a) a scree plot of the explained variance ratio vs PC index (log y-axis), and (b) a plot of the first three PC basis vectors $\phi_k$ as functions of $\log_{10} M$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(np.arange(1, n_pc + 1), pca.explained_variance_ratio_, 'o-')
axes[0].set_xlabel('PC index'); axes[0].set_ylabel('explained variance ratio'); axes[0].set_title('Scree plot')
for k in range(min(3, n_pc)):
    axes[1].plot(log10M_mid, pca.components_[k], label=f'PC{k+1}')
axes[1].set_xlabel(r'$\log_{10} M$'); axes[1].set_ylabel('loading'); axes[1].legend(); axes[1].set_title('PCA basis vectors')
for ax in axes: ax.grid(True, which='both', ls=':')

### Exercise 4.2 — Reconstruction check
Take training sample `j = 0`, reconstruct its HMF from its PC scores (`pca.inverse_transform` → `scaler.inverse_transform` → `10**`), and plot the fractional difference relative to the truth. Repeat using only the first 1, 2, 3 components (set the remaining scores to zero).


In [ ]:
def reconstruct_hmf(scores):
    """Map PC scores (shape (n_pc,)) back to N_halo per bin."""
    y_scaled = pca.inverse_transform(np.atleast_2d(scores))
    return 10 ** scaler.inverse_transform(y_scaled).ravel()

j = 0
for k in [1, 2, 3, n_pc]:
    truncated = np.zeros(n_pc)
    truncated[:k] = pc_scores[j, :k]
    rec = reconstruct_hmf(truncated)
    plt.plot(log10M_mid, np.abs(rec / hmfs[j] - 1), label=f'{k} PCs')

plt.yscale('log'); plt.xlabel(r'$\log_{10} M$'); plt.ylabel('|fractional error|'); plt.legend()

**Q4.2** How many components do you need for a reconstruction error below 0.1%? Does this match the explained-variance numbers? Note that this is a *training* sample; the emulator will also have *interpolation* error on top of this.

*Your answer:*


---
## Part 5 — Gaussian Process regression

A Gaussian Process defines a distribution over functions. Given training data $(\mathbf{x}_i, y_i)$ and a **kernel** $k(\mathbf{x}, \mathbf{x}')$ that encodes how correlated outputs are for nearby inputs, a GP gives at any new $\mathbf{x}_*$ a predictive **mean** and **uncertainty**. We use the RBF (squared-exponential) kernel with one length-scale per input dimension:

$$ k(\mathbf{x},\mathbf{x}') = \sigma^2 \exp\left(-\sum_d \frac{(x_d - x'_d)^2}{2\ell_d^2}\right) + \sigma_n^2\,\delta_{\mathbf{x}\mathbf{x}'} .$$

The hyperparameters $(\sigma, \ell_d, \sigma_n)$ are found by maximising the marginal likelihood. We fit **one independent GP per PC coefficient**.

It is good practice to rescale the inputs to $[0, 1]$ so the length-scales are comparable.


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

def to_unit(params):
    """Map parameters from `bounds` to the unit cube."""
    return (params - bounds[:, 0]) / (bounds[:, 1] - bounds[:, 0])

X_gp = to_unit(train_params)

kernel = (C(1.0, (1e-3, 1e3))
          * RBF(length_scale=np.ones(n_params), length_scale_bounds=(1e-2, 1e2))
          + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1)))

gp_models = []
for k in range(n_pc):
    gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                   n_restarts_optimizer=5, random_state=42)
    gpr.fit(X_gp, pc_scores[:, k])
    gp_models.append(gpr)
    print(f"PC{k+1}: {gpr.kernel_}")

**Q5.1** Look at the fitted length-scales for PC1. Is the function smoother along $\Omega_m$ or along $\ln A_s$ (in unit-cube coordinates)? How do the length-scales change for higher PCs, and what does that tell you?

*Your answer:*


### Exercise 5.1 — Visualise the GP surfaces
Evaluate the GP mean **and standard deviation** for each PC on a $50\times50$ grid over the parameter box. Plot the mean as a filled contour with the training points overlaid, and the standard deviation in a second row.


In [ ]:
n_grid = 50
om_grid = np.linspace(*bounds[0], n_grid)
lnA_grid = np.linspace(*bounds[1], n_grid)
Om, LnA = np.meshgrid(om_grid, lnA_grid)
grid_params = np.column_stack([Om.ravel(), LnA.ravel()])

means, stds = [], []
for gpr in gp_models:
    mu, sd = gpr.predict(to_unit(grid_params), return_std=True)
    means.append(mu.reshape(n_grid, n_grid)); stds.append(sd.reshape(n_grid, n_grid))

fig, axes = plt.subplots(2, n_pc, figsize=(4.5 * n_pc, 8), squeeze=False)
for k in range(n_pc):
    m = axes[0, k].contourf(om_grid, lnA_grid, means[k], levels=30, cmap='viridis')
    axes[0, k].scatter(train_params[:, 0], train_params[:, 1], c=pc_scores[:, k], cmap='viridis', edgecolor='k', s=25)
    plt.colorbar(m, ax=axes[0, k]); axes[0, k].set_title(f'PC{k+1} GP mean')
    s = axes[1, k].contourf(om_grid, lnA_grid, stds[k], levels=30, cmap='magma')
    axes[1, k].plot(train_params[:, 0], train_params[:, 1], 'w.', ms=3)
    plt.colorbar(s, ax=axes[1, k]); axes[1, k].set_title(f'PC{k+1} GP std')
    for r in range(2):
        axes[r, k].set_xlabel(r'$\Omega_m$'); axes[r, k].set_ylabel(r'$\ln A_s$')
plt.tight_layout()

**Q5.2** Where is the GP uncertainty largest? Why?

*Your answer:*


---
## Part 6 — Putting it together: the emulator

Wrap everything in a single function `emulate_hmf(Om, lnAs)` that
1. maps the inputs to the unit cube,
2. predicts the PC scores with the GPs,
3. reconstructs $N_{\rm halo}$ with `reconstruct_hmf`.

Then test it at a random point that was **not** in the training set.


In [ ]:
def emulate_hmf(Om, lnAs):
    x = to_unit(np.array([[Om, lnAs]]))
    scores = np.array([gpr.predict(x)[0] for gpr in gp_models])
    return reconstruct_hmf(scores)

rng = np.random.default_rng(12345)
Om_t, lnAs_t = rng.uniform(bounds[:, 0], bounds[:, 1])
print(f"test point: Omega_m={Om_t:.4f}, lnAs={lnAs_t:.4f}")

hmf_true = get_hmf_emu(Om_t, lnAs_t)
hmf_emu = emulate_hmf(Om_t, lnAs_t)
frac_err = hmf_emu / hmf_true - 1

fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axes[0].semilogy(log10M_mid, hmf_true, '-', label='truth (Dark Emulator)')
axes[0].semilogy(log10M_mid, hmf_emu, '--', label='GP+PCA emulator')
axes[0].legend(); axes[0].set_ylabel(r'$N_{\rm halo}$')
axes[1].plot(log10M_mid, 100 * frac_err, 'o-'); axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_xlabel(r'$\log_{10} M$'); axes[1].set_ylabel('error [%]')
for ax in axes: ax.grid(True, which='both', ls=':')
print(f"max |error| = {100*np.max(np.abs(frac_err)):.3f} %")

---
## Part 7 — Systematic validation

A single test point can be lucky. Build an independent **validation set** of 40 random points, compute the fractional error for each, and show the distribution of errors as a function of mass (e.g. the median and 68% / 95% bands of $|\epsilon|$).


In [ ]:
n_val = 40
rng = np.random.default_rng(2026)
val_params = rng.uniform(bounds[:, 0], bounds[:, 1], size=(n_val, 2))

errs = np.array([emulate_hmf(*p) / get_hmf_emu(*p) - 1 for p in val_params])

abs_err = 100 * np.abs(errs)
p50, p68, p95 = np.percentile(abs_err, [50, 68, 95], axis=0)
plt.fill_between(log10M_mid, 0, p95, alpha=0.2, label='95%')
plt.fill_between(log10M_mid, 0, p68, alpha=0.4, label='68%')
plt.plot(log10M_mid, p50, 'k-', label='median')
plt.yscale('log'); plt.xlabel(r'$\log_{10} M$'); plt.ylabel('|error| [%]'); plt.legend(); plt.grid(True, which='both', ls=':')
print(f"median |error| = {100*np.median(np.abs(errs)):.3f} %,  max |error| = {100*np.max(np.abs(errs)):.3f} %")

**Q7.1** Where in mass is the emulator least accurate? Relate this to your answer to Q1.1 and to the PCA reconstruction test.

*Your answer:*


---
## Part 8 — Improving the emulator (open-ended)

To make the experiments easy, here is a helper that rebuilds the *whole* pipeline from a training design and returns the validation error on `val_params`. Fill in the blanks, then use it for the exercises below.


In [ ]:
def build_and_validate(train_params, n_pc=None, var_frac=0.9999, log_space=True, kernel=None, verbose=False):
    """Train a GP+PCA emulator and return fractional errors on val_params, shape (n_val, nbin)."""
    Y = np.array([get_hmf_emu(*p) for p in train_params])
    if log_space:
        Y = np.log10(Y)
    sc = StandardScaler().fit(Y)
    pc = PCA(n_components=n_pc if n_pc is not None else var_frac, svd_solver='full')
    S = pc.fit_transform(sc.transform(Y))
    if kernel is None:
        kernel = (C(1.0, (1e-3, 1e3)) * RBF(np.ones(n_params), (1e-2, 1e2))
                  + WhiteKernel(1e-6, (1e-10, 1e-1)))
    gps = [GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3,
                                    random_state=42).fit(to_unit(train_params), S[:, k])
           for k in range(S.shape[1])]

    def emu_fn(p):
        x = to_unit(np.atleast_2d(p))
        scores = np.array([[g.predict(x)[0] for g in gps]])
        y = sc.inverse_transform(pc.inverse_transform(scores)).ravel()
        return 10 ** y if log_space else y

    errs = np.array([emu_fn(p) / get_hmf_emu(*p) - 1 for p in val_params])
    if verbose:
        print(f"n_train={len(train_params)}, n_pc={S.shape[1]}: median={100*np.median(np.abs(errs)):.3f}%, max={100*np.max(np.abs(errs)):.3f}%")
    return errs

### Exercise 8.1 — Number of PCA components
Run `build_and_validate` with `n_pc = 1, 2, 3, 4, 6` (using the 100-point LHS design). Plot the maximum and median validation error vs `n_pc`. Is there a point beyond which adding components stops helping? Why might adding *too many* components hurt?


In [ ]:
res = {}
for n in [1, 2, 3, 4, 6]:
    e = build_and_validate(train_params, n_pc=n, verbose=True)
    res[n] = (np.median(np.abs(e)), np.max(np.abs(e)))
ns = list(res)
plt.semilogy(ns, [100*res[n][0] for n in ns], 'o-', label='median')
plt.semilogy(ns, [100*res[n][1] for n in ns], 's-', label='max')
plt.xlabel('number of PCs'); plt.ylabel('validation |error| [%]'); plt.legend(); plt.grid(True, which='both', ls=':')

### Exercise 8.2 — Size of the training set
Generate LHS designs with 10, 20, 50, 100, 200 points and plot the validation error vs training-set size. How does the error scale? (Remember: in a real application each training point is an expensive simulation!)


In [ ]:
res = {}
for n_tr in [10, 20, 50, 100, 200]:
    design = qmc.scale(qmc.LatinHypercube(d=n_params, seed=1).random(n_tr), bounds[:, 0], bounds[:, 1])
    e = build_and_validate(design, verbose=True)
    res[n_tr] = (np.median(np.abs(e)), np.max(np.abs(e)))
ns = list(res)
plt.loglog(ns, [100*res[n][0] for n in ns], 'o-', label='median')
plt.loglog(ns, [100*res[n][1] for n in ns], 's-', label='max')
plt.xlabel('training-set size'); plt.ylabel('validation |error| [%]'); plt.legend(); plt.grid(True, which='both', ls=':')

### Exercise 8.3 — Log vs linear
Repeat the 100-point run with `log_space=False`. Compare the error as a function of mass to the log-space result. Explain the difference.


In [ ]:
e_log = build_and_validate(train_params, log_space=True, verbose=True)
e_lin = build_and_validate(train_params, log_space=False, verbose=True)
plt.semilogy(log10M_mid, 100*np.max(np.abs(e_log), axis=0), label='log space')
plt.semilogy(log10M_mid, 100*np.max(np.abs(e_lin), axis=0), label='linear space')
plt.xlabel(r'$\log_{10} M$'); plt.ylabel('max |error| [%]'); plt.legend(); plt.grid(True, which='both', ls=':')

### Exercise 8.4 — Kernel choice
Try a `Matern` kernel (`from sklearn.gaussian_process.kernels import Matern`) with `nu = 1.5` and `nu = 2.5` instead of the RBF. Which gives the smallest validation error? Look up what $\nu$ controls.


In [ ]:
from sklearn.gaussian_process.kernels import Matern
for nu in [1.5, 2.5]:
    kern = C(1.0, (1e-3, 1e3)) * Matern(np.ones(n_params), (1e-2, 1e2), nu=nu) + WhiteKernel(1e-6, (1e-10, 1e-1))
    print(f"Matern nu={nu}:", end=' ')
    build_and_validate(train_params, kernel=kern, verbose=True)
print("RBF:", end=' ')
build_and_validate(train_params, verbose=True)

### Exercise 8.5 — Extrapolation (challenge)
Evaluate your emulator *outside* the training box (e.g. $\Omega_m = 0.45$, $\ln A_s = 3.1$). How large is the error? What does the GP standard deviation say? Discuss why emulators should never be trusted outside their training domain.


In [ ]:
for Om_x in [0.4, 0.42, 0.45, 0.5]:
    x = to_unit(np.array([[Om_x, 3.1]]))
    sds = [g.predict(x, return_std=True)[1][0] for g in gp_models]
    err = emulate_hmf(Om_x, 3.1) / get_hmf_emu(Om_x, 3.1) - 1
    print(f"Omega_m={Om_x:.2f}: max |error| = {100*np.max(np.abs(err)):7.3f} %   GP std per PC = {np.round(sds, 3)}")

---
## Part 9 — Summary and further reading

You have built an emulator that reproduces a (here pretend-)expensive calculation to sub-percent accuracy across a 2D parameter space using only 100 training evaluations. The same recipe — **space-filling design → dimensionality reduction → GP per coefficient → validation** — underlies real-world cosmological emulators.

**Bonus project ideas**
- Add a third parameter (e.g. $n_s$ or $w$) and see how the required training-set size grows.
- Emulate the HMF at several redshifts jointly (treat $z$ as an extra input).
- Use the GP predictive variance to choose where to add new training points (*active learning*).
- Replace the GP by a small neural network and compare accuracy and speed.

**References**
- Rasmussen & Williams, *Gaussian Processes for Machine Learning* (free online: gaussianprocess.org/gpml)
- Heitmann et al. 2009, "The Coyote Universe I" — the original cosmic emulator paper
- Nishimichi et al. 2019, "Dark Quest I" — the Dark Emulator paper
- scikit-learn user guide, section 1.7 *Gaussian Processes*
